# Cobrinha — do código à interação
Aprenda a construir Cobrinha com o mesmo código Python do projeto interativo. A sequência ensina estado, regras e interface. Cada trecho é parte do programa final; os recuos devem ser mantidos. A instalação vai em uma célula separada e o programa completo em outra.

In [ ]:
%pip install -q "ipywidgets>=8.1,<9"


## Criar o estado da cobra
import random permite sortear o alimento. A classe Cobrinha reúne o estado e as regras. __init__ cria o corpo, a direção, os pontos e o estado de fim de jogo. self identifica a partida atual. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
import random

class Cobrinha:
    def __init__(self, tamanho=16):
        self.tamanho = tamanho
        self.corpo = [(5, 7), (4, 7), (3, 7)]
        self.direcao = (1, 0)
        self.proxima = self.direcao
        self.pontos = 0
        self.fim = False
        self.comida = self.sortear_comida()


```

Por que cada parte do corpo é uma tupla (x, y)?

## Sortear alimento em uma casa livre
A compreensão de lista enumera as casas do tabuleiro e exclui as ocupadas pela cobra. random.choice escolhe uma casa livre. Se não houver nenhuma, a função retorna None. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def sortear_comida(self):
        livres = [(x, y) for x in range(self.tamanho) for y in range(self.tamanho)
                  if (x, y) not in self.corpo]
        return random.choice(livres) if livres else None


```

Por que o alimento não pode aparecer dentro da cobra?

## Controlar a direção
virar recebe dx e dy. A condição aceita somente as quatro direções e rejeita a direção oposta à direção do movimento atual. proxima guarda a mudança para o próximo passo. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def virar(self, dx, dy):
        if (dx, dy) in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            if (dx, dy) != (-self.direcao[0], -self.direcao[1]):
                self.proxima = (dx, dy)


```

O que aconteceria se a cobra pudesse inverter a direção imediatamente?

## Mover, crescer e detectar colisões
passo calcula a próxima cabeça. Se ela alcançar comida, o corpo cresce; caso contrário, a cauda sai. Testamos limites e corpo antes de inserir a cabeça. A cauda que vai sair não conta como obstáculo. Ao comer, somamos um ponto e sorteamos outro alimento. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def passo(self):
        if self.fim:
            return
        self.direcao = self.proxima
        x, y = self.corpo[0]
        cabeca = (x + self.direcao[0], y + self.direcao[1])
        cresce = cabeca == self.comida
        obstaculos = self.corpo if cresce else self.corpo[:-1]
        if not (0 <= cabeca[0] < self.tamanho and 0 <= cabeca[1] < self.tamanho) or cabeca in obstaculos:
            self.fim = True
            return
        self.corpo.insert(0, cabeca)
        if cresce:
            self.pontos += 1
            self.comida = self.sortear_comida()
            if self.comida is None:
                self.fim = True
        else:
            self.corpo.pop()


```

Em qual situação devemos evitar corpo.pop()?

## Preparar a interface
Este bloco prepara asyncio, ipywidgets e display no notebook. Cancela o temporizador de uma execução anterior, cria uma partida e começa pausado. HTML mostra o tabuleiro; Button cria os controles. Todo o bloco permanece com quatro espaços de recuo. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
if __name__ == "__main__":
    import asyncio
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass

    # Encerra a partida anterior quando a célula é executada novamente.
    if "tarefa_cobrinha" in globals():
        tarefa_cobrinha.cancel()
    jogo = Cobrinha()
    rodando = False
    painel = w.HTML()
    iniciar = w.Button(description="Iniciar / Pausar")
    reiniciar = w.Button(description="Reiniciar")


```

Por que cancelar o temporizador ao executar a célula novamente?

## Desenhar o tabuleiro
desenhar constrói um SVG: cada segmento vira um retângulo e a comida vira um círculo. Multiplicar as coordenadas por 20 converte casas em pixels. painel.value atualiza a saída sem abrir outra janela. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def desenhar():
        blocos = ""
        for i, (x, y) in enumerate(jogo.corpo):
            cor = "#a3e635" if i == 0 else "#22c55e"
            blocos += f'<rect x="{x*20+1}" y="{y*20+1}" width="18" height="18" rx="4" fill="{cor}"/>'
        if jogo.comida is not None:
            x, y = jogo.comida
            blocos += f'<circle cx="{x*20+10}" cy="{y*20+10}" r="7" fill="#fb7185"/>'
        estado = "Fim de jogo — Reinicie" if jogo.fim else ("Jogando" if rodando else "Pausado — clique em Iniciar")
        painel.value = f'<p><b>Cobrinha | Pontos: {jogo.pontos}</b> | {estado}</p><svg viewBox="0 0 320 320" width="320" style="max-width:100%;background:#0f172a">{blocos}</svg>'


```

Qual é a diferença entre a coordenada de uma casa e sua posição em pixels?

## Ligar os botões às regras
alternar muda rodando; resetar cria outra partida. on_click registra funções, sem executá-las naquele momento. Os valores dx=dx e dy=dy preservam a direção de cada botão criado no laço. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def alternar(_):
        global rodando
        if not jogo.fim:
            rodando = not rodando
        desenhar()

    def resetar(_):
        global jogo, rodando
        jogo = Cobrinha()
        rodando = False
        desenhar()

    iniciar.on_click(alternar)
    reiniciar.on_click(resetar)
    botoes = []
    for texto, dx, dy in [("↑", 0, -1), ("←", -1, 0), ("↓", 0, 1), ("→", 1, 0)]:
        b = w.Button(description=texto, layout=w.Layout(width="65px", height="45px"))
        b.on_click(lambda _, dx=dx, dy=dy: jogo.virar(dx, dy))
        botoes.append(b)


```

Por que não chamamos a função com parênteses ao registrar on_click?

## Atualizar a partida sem bloquear
O laço espera 0.28 segundo usando await. Quando a partida está rodando, calcula um passo e redesenha. display mostra os widgets e create_task agenda o laço no ambiente do notebook. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    async def atualizar():
        while True:
            await asyncio.sleep(0.28)
            if rodando and not jogo.fim:
                jogo.passo()
                desenhar()

    display(w.VBox([painel, w.HBox(botoes), w.HBox([iniciar, reiniciar])]))
    desenhar()
    tarefa_cobrinha = asyncio.get_running_loop().create_task(atualizar())

```

Qual é o efeito de aumentar o tempo de asyncio.sleep?

## Executar o programa completo
Este é exatamente o programa montado nas etapas anteriores e usado no notebook vinculado. Depois da instalação, copie este bloco inteiro para outra célula e execute. Clique em Iniciar / Pausar e use os botões. Mantenha a sessão do Colab conectada.

In [ ]:
import random

class Cobrinha:
    def __init__(self, tamanho=16):
        self.tamanho = tamanho
        self.corpo = [(5, 7), (4, 7), (3, 7)]
        self.direcao = (1, 0)
        self.proxima = self.direcao
        self.pontos = 0
        self.fim = False
        self.comida = self.sortear_comida()

    def sortear_comida(self):
        livres = [(x, y) for x in range(self.tamanho) for y in range(self.tamanho)
                  if (x, y) not in self.corpo]
        return random.choice(livres) if livres else None

    def virar(self, dx, dy):
        if (dx, dy) in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            if (dx, dy) != (-self.direcao[0], -self.direcao[1]):
                self.proxima = (dx, dy)

    def passo(self):
        if self.fim:
            return
        self.direcao = self.proxima
        x, y = self.corpo[0]
        cabeca = (x + self.direcao[0], y + self.direcao[1])
        cresce = cabeca == self.comida
        obstaculos = self.corpo if cresce else self.corpo[:-1]
        if not (0 <= cabeca[0] < self.tamanho and 0 <= cabeca[1] < self.tamanho) or cabeca in obstaculos:
            self.fim = True
            return
        self.corpo.insert(0, cabeca)
        if cresce:
            self.pontos += 1
            self.comida = self.sortear_comida()
            if self.comida is None:
                self.fim = True
        else:
            self.corpo.pop()

if __name__ == "__main__":
    import asyncio
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass

    # Encerra a partida anterior quando a célula é executada novamente.
    if "tarefa_cobrinha" in globals():
        tarefa_cobrinha.cancel()
    jogo = Cobrinha()
    rodando = False
    painel = w.HTML()
    iniciar = w.Button(description="Iniciar / Pausar")
    reiniciar = w.Button(description="Reiniciar")

    def desenhar():
        blocos = ""
        for i, (x, y) in enumerate(jogo.corpo):
            cor = "#a3e635" if i == 0 else "#22c55e"
            blocos += f'<rect x="{x*20+1}" y="{y*20+1}" width="18" height="18" rx="4" fill="{cor}"/>'
        if jogo.comida is not None:
            x, y = jogo.comida
            blocos += f'<circle cx="{x*20+10}" cy="{y*20+10}" r="7" fill="#fb7185"/>'
        estado = "Fim de jogo — Reinicie" if jogo.fim else ("Jogando" if rodando else "Pausado — clique em Iniciar")
        painel.value = f'<p><b>Cobrinha | Pontos: {jogo.pontos}</b> | {estado}</p><svg viewBox="0 0 320 320" width="320" style="max-width:100%;background:#0f172a">{blocos}</svg>'

    def alternar(_):
        global rodando
        if not jogo.fim:
            rodando = not rodando
        desenhar()

    def resetar(_):
        global jogo, rodando
        jogo = Cobrinha()
        rodando = False
        desenhar()

    iniciar.on_click(alternar)
    reiniciar.on_click(resetar)
    botoes = []
    for texto, dx, dy in [("↑", 0, -1), ("←", -1, 0), ("↓", 0, 1), ("→", 1, 0)]:
        b = w.Button(description=texto, layout=w.Layout(width="65px", height="45px"))
        b.on_click(lambda _, dx=dx, dy=dy: jogo.virar(dx, dy))
        botoes.append(b)

    async def atualizar():
        while True:
            await asyncio.sleep(0.28)
            if rodando and not jogo.fim:
                jogo.passo()
                desenhar()

    display(w.VBox([painel, w.HBox(botoes), w.HBox([iniciar, reiniciar])]))
    desenhar()
    tarefa_cobrinha = asyncio.get_running_loop().create_task(atualizar())
